# Corporate Reputation: Run all five extractors

**One Google Drive mount • One ticker • One inclusive date range • Separate source outputs.**

1. Copy the *contents* of the framework ZIP into `MyDrive/ColabNotebooks/DataForReputation/`.
2. For YouTube, add Colab Secret `YOUTUBE_API_KEY` with notebook access enabled. For SEC, add `SEC_USER_AGENT` (descriptive identity + real email).
3. Run cells top-to-bottom. Review `run_summary.csv` and `manifest.json` for partial or failed sources.
4. The framework NEVER sends raw datasets to GitHub; see the other notebook for code-only publishing.

**Research caution:** YouTube search finds videos published within the period, then filters their comments by publication date. SEC filings indicate disclosures, not necessarily a reputational event. Yahoo stock bars are daily, while Google Trends frequency varies.

In [1]:
# 1. Mount Drive ONCE; point Python at the framework folder.
from google.colab import drive
from pathlib import Path
import sys

drive.mount('/content/drive')
PROJECT_ROOT = Path('/content/drive/MyDrive/ColabNotebooks/DataForReputation')
if not (PROJECT_ROOT / 'reputation_etl').is_dir():
    raise FileNotFoundError(f'Copy the framework files to {PROJECT_ROOT} first')
sys.path.insert(0, str(PROJECT_ROOT))
print('Framework folder:', PROJECT_ROOT)

Mounted at /content/drive
Framework folder: /content/drive/MyDrive/ColabNotebooks/DataForReputation


In [ ]:
# 2. Install required packages. Run once after each new Colab runtime starts.
import subprocess
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       '-r', str(PROJECT_ROOT / 'requirements.txt')])

0

## 3. Parameters

Edit these values once. If you lack access to an API, remove that source from `SOURCES` until configured. No keys are typed into code cells. If the company-name lookup is unreliable, provide a verified name override.

In [37]:
from google.colab import userdata
from reputation_etl import ExtractionConfig, run_pipeline

def get_secret(name):
    try:
        return userdata.get(name)
    except Exception:
        return None

TICKER = 'MSFT'
START_DATE = '2026-10-03'  # inclusive
END_DATE = '2026-10-03'    # inclusive
COMPANY_NAME_OVERRIDE = 'Microsoft Inc.'  # None to resolve via yfinance

# Remove a source if its credentials/approval are not available.
SOURCES = ('gdelt', 'google_trends', 'youtube', 'sec_edgar', 'yahoo_finance')

config = ExtractionConfig(
    ticker=TICKER, start_date=START_DATE, end_date=END_DATE,
    sources=SOURCES, output_root=PROJECT_ROOT,
    company_name_override=COMPANY_NAME_OVERRIDE,
    gdelt_aliases=('Microsoft',) if TICKER == 'MSFT' else (),
    gdelt_window_days=1,
    trends_geo='US',
    trends_search_term=None,
    youtube_api_key=get_secret('YOUTUBE_API_KEY'),
    #youtube_api_key=get_secret('GITHUB_TOKEN'),
    youtube_max_videos=50,
    youtube_max_comment_pages_per_video=10,
    youtube_include_comment_text=False, # must explicitly opt-in if required
    youtube_include_author_fields=False,
    sec_user_agent=get_secret('SEC_USER_AGENT'),
)
print('Ticker:', config.ticker, '| dates:', config.start_date, 'to', config.end_date)
print('Selected:', ', '.join(config.sources))
print('YouTube key configured:', bool(config.youtube_api_key))
print('SEC user agent configured:', bool(config.sec_user_agent))

Ticker: MSFT | dates: 2026-10-03 to 2026-10-03
Selected: gdelt, google_trends, youtube, sec_edgar, yahoo_finance
YouTube key configured: False
SEC user agent configured: False


In [38]:
# 4. Run extraction. Sources fail independently; inspect statuses afterward.
summary, run_directory = run_pipeline(config)
display(summary)
print('Saved to:', run_directory)

Company: MSFT — Microsoft Inc.

Running gdelt ...
gdelt: empty — 0 rows

Running google_trends ...


ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['MSFT']: YFPricesMissingError('possibly delisted; no price data found  (1d 2026-10-03 -> 2026-10-04)')


google_trends: success — 1 rows

Running youtube ...
youtube: failed — 0 rows; ValueError: Missing YouTube API key; add YOUTUBE_API_KEY to Colab Secrets

Running sec_edgar ...
sec_edgar: failed — 0 rows; ValueError: SEC user_agent must identify your research and include a real contact email

Running yahoo_finance ...
yahoo_finance: empty — 0 rows

Outputs: /content/drive/MyDrive/ColabNotebooks/DataForReputation/runs/MSFT/2026-10-03_to_2026-10-03/20261101T194844Z_218dcbdd


,source,status,rows,error,notes
0,gdelt,empty,0,,GDELT DOC ArticleList is capped at 250 article...
1,google_trends,success,1,,Google Trends values are normalized within the...
2,youtube,failed,0,ValueError: Missing YouTube API key; add YOUTU...,
3,sec_edgar,failed,0,ValueError: SEC user_agent must identify your ...,
4,yahoo_finance,empty,0,,yfinance end parameter converted to inclusive ...


Saved to: /content/drive/MyDrive/ColabNotebooks/DataForReputation/runs/MSFT/2026-10-03_to_2026-10-03/20261101T194844Z_218dcbdd


In [35]:
# 5. Inspect saved files and manifest (no API keys stored in manifest).
import json
manifest = json.loads((run_directory / 'manifest.json').read_text())
for source, info in manifest['sources'].items():
    print(f"{source:16} {info['status']:8} rows={info['rows']:>6}")
    for note in info.get('notes', []):
        print('   NOTE:', note)
print('\nOutput files:')
for file in sorted(run_directory.iterdir()):
    print(' -', file.name)

gdelt            success  rows=   183
   NOTE: GDELT DOC ArticleList is capped at 250 articles per request and has limited historical access; capped windows or unavailable periods do not imply no articles exist.
google_trends    success  rows=     1
   NOTE: Google Trends values are normalized within the request; do not compare uncalibrated runs.
   NOTE: Output observation cadence is determined by the requested historical range, not guaranteed hourly.
   NOTE: pytrends is an unofficial archived client; access may fail or change.
youtube          failed   rows=     0
sec_edgar        failed   rows=     0
yahoo_finance    success  rows=     1
   NOTE: yfinance end parameter converted to inclusive end date.
   NOTE: Daily stock bars are not hourly prediction signals; market data are external corroboration.
   NOTE: Rolling 7/30-trading-day variables are NaN near the start unless prior warm-up history is fetched.
   NOTE: yfinance is unofficial; verify data license and replace with licens

## Before interpreting the data

A **partial** status means a source may be incomplete. An **empty** dataset is not proof of no reputational activity. Always inspect API constraints, source timestamps, selection biases, and the request logs. Preserve raw native-time observations before aggregating to hourly features; stock prices and historical search interest do not necessarily have hourly cadence.